#### Imports

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


import torch
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import pytorch_lightning as pl

import wandb
wandb.login()


from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar, ModelCheckpoint
from pytorch_lightning.tuner import Tuner

from utils import seed_everything, VolumeDataModule3D
from models_3d import PredFormer, Pl_Model

cuda


wandb: Currently logged in as: nbennewiz to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
seed_everything(100)

Seed set to 100


100

#### Config

In [3]:
config = {
    #for the dataloaders
    "root": "../NormalizedQualityFiltered",
    'batch_size': 1,
    'learning_rate': 1e-4,
    "num_workers": 20,#0, wenn die gpu nicht benutzt wird
    "pin_memory": True if torch.cuda.is_available() else False,#False, wenn die gpu nicht benutzt wird
    "drop_last": False,
    'epochs': 100,
    #'log_interval': 20,
    #'viz_interval': 1,
    'run_name': '3D-SimVP_big_huberssim',
    'input_frames': 9,
    "pred_frames": 9,
    "pred_n_frames_per_step": 9,
    'base_filters': 32,
    "train_split": 0.7,
    "val_split": 0.15,
    "test_split": 0.15,
}
config["run_name"] += f"_{config['pred_frames']}"
if config["pred_frames"] == config["pred_n_frames_per_step"]:
    config["run_name"] += "_NAR"
elif config["pred_n_frames_per_step"] == 1:
    config["run_name"] += "_FAR"
else:
    config["run_name"] += f"_PAR_{config['pred_n_frames_per_step']}"


# Get data module
dm = VolumeDataModule3D(
    root=config["root"],
    batch_size=config['batch_size'],
    num_workers=config["num_workers"],
    pin_memory=config["pin_memory"],
    drop_last=config["drop_last"],
    sequence_length=config["input_frames"],
    prediction_length=config["pred_frames"],
    train_split=config["train_split"],
    val_split=config["val_split"],
    test_split=config["test_split"],
)
wandb_logger = WandbLogger(entity="ChadCTP", project="perfusion-ct-prediction", name=config["run_name"])

# Initialize model for tuning
model_config = {
    # image h w c
    'image_depth': 16,
    'height': 256,
    'width': 256,
    'num_channels': 1,
    # video length in and out
    'pre_seq': config["input_frames"],
    'after_seq': config["pred_n_frames_per_step"],
    # patch size
    'patch_size': 8,
    'dim': 256, 
    'heads': 8,
    'dim_head': 32,
    # dropout
    'dropout': 0.1,
    'attn_dropout': 0.1,
    'drop_path': 0.25,
    'scale_dim': 2,
    # depth
    'depth': 1,
    'Ndepth': 2, # For FullAttention-8, for BinaryST, BinaryST, FacST, FacTS-4, for TST,STS-3, for TSST, STTS-2
}
model = PredFormer(model_config=model_config)

# Initialize pl_model for tuning
pl_model = Pl_Model(
    passed_model=model,
    config=config,
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_total_loss",  
    mode="min",  
    save_top_k=1,
    filename="best-checkpoint",
    verbose=True,
)

# Initialize trainer for tuning
trainer = pl.Trainer(
    logger=wandb_logger,
    accelerator="gpu",
    devices= [0] if torch.cuda.is_available() else None,
    max_epochs=config["epochs"],
    callbacks=[RichProgressBar(), checkpoint_callback],
    check_val_every_n_epoch=5,
    precision="bf16-mixed",
)

#tuning
#tuner = Tuner(trainer)
#tuner.scale_batch_size(pl_model, datamodule=dm, mode="binsearch")

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [5]:
trainer.fit(
    model=pl_model,
    datamodule=dm,
)

Epoch 0/99 ━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━ 25/83 0:00:10 • 0:00:24 2.47it/s v_num: 3lpk train_total_loss: 0.314


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [5]:
#check and log the losses "to beat"
dm.setup()
pl_model.check_losses(dm.train_dataloader(), mode="train", use_wandb=True)
pl_model.check_losses(dm.val_dataloader(), mode="val", use_wandb=True)
pl_model.check_losses(dm.test_dataloader(), mode="test", use_wandb=True)

test_results = trainer.test(pl_model, datamodule=dm)

save_load_path = f"../ModelWeights/{config['run_name']}.ckpt"
trainer.save_checkpoint(save_load_path)

Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 0:00:22 • 0:00:00 0.83it/s

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      test_huber_loss      │   0.014533606357872486    │
│    test_huberssim_loss    │    0.3081701099872589     │
│       test_mse_loss       │     0.111053965985775     │
│      test_psnr_loss       │     9.545494079589844     │
│      test_rmse_loss       │    0.33323097229003906    │
│      test_ssim_loss       │    0.9895232319831848     │
│    test_temporal_loss     │   0.011394299566745758    │
│      test_total_loss      │    0.3081701099872589     │
└───────────────────────────┴───────────────────────────┘

@torch.no_grad()
def overall_loss(model, loader, device):
    mse_loss = 0.0
    huber_loss = 0.0
    rmse_loss = 0.0
    #ssim_loss = 0.0
    psnr_loss = 0.0
    total_loss = 0.0
    model = model.to(device)
    for inputs, targets in loader:
        print(targets.shape)
        inputs = inputs.to(device)
        targets = targets.to(device)
        outputs = []
        for t in range(0, model.config["pred_frames"], model.config["pred_n_frames_per_step"]):
            if model.config["pred_frames"]-t<model.config["pred_n_frames_per_step"]:
                frames_this_step = model.config["pred_frames"]-t
            else:
                frames_this_step = model.config["pred_n_frames_per_step"]
            outputs_t = model.forward(inputs)
            #print(f"{t}:{t+frames_this_step}")
            #get only the first predicted frame
            outputs_t = outputs_t[:, :frames_this_step, :, :, :, :]
            
            outputs.append(outputs_t)

            inputs = torch.cat([inputs[:, model.config["pred_n_frames_per_step"]:, :, :, :, :], outputs_t], dim=1)
            
            #concat time and add to overall lst
        outputs = torch.concat(outputs, dim=1)
        
        #calculate losses
        mse_loss += model.mse_criterion(outputs, targets).item()
        huber_loss += model.huber_criterion(outputs, targets).item()
        rmse_loss += torch.sqrt(model.mse_criterion(outputs, targets)).item()
        #ssim_loss = model.ssim_criterion(outputs, targets).item()
        psnr_loss += model.psnr_criterion(outputs, targets).item()
        total_loss += mse_loss + 0.5 * huber_loss

    mse_loss = mse_loss / len(loader)
    huber_loss = huber_loss / len(loader)
    rmse_loss = rmse_loss / len(loader)
    #ssim_loss = ssim_loss / len(loader)
    psnr_loss = psnr_loss / len(loader)
    total_loss = total_loss / len(loader)

    return outputs, mse_loss, huber_loss, rmse_loss, psnr_loss, total_loss

dm.setup()
_, mse_loss, huber_loss, rmse_loss, psnr_loss, total_loss = overall_loss(model=pl_model, loader=dm.test_dataloader(), device=device)
mse_loss, huber_loss, rmse_loss, psnr_loss, total_loss



#testing
pl_model = Pl_Model.load_from_checkpoint(
    "./perfusion-ct-prediction/5iq01p4o/checkpoints/best-checkpoint.ckpt",
    passed_model=model,
)

dm.setup()
test_dataloader = dm.test_dataloader()

print(len(test_dataloader))
pl_model.to(device)
"""for inputs, targets in test_dataloader:
    outputs = pl_model.forward(inputs.to(device))
    outputs = outputs.detach().cpu()
    outputs = torch.concat([inputs, outputs], dim=1).squeeze(0, 2).numpy()
    targets = torch.concat([inputs, targets], dim=1).squeeze(0, 2).numpy()
    print(outputs.shape)
    print(targets.shape)
    break"""
# method 1
#inputs, targets = next(iter(test_dataloader))

# method 2
# only for single shot
file_name = "MOL-253"
vol = torch.tensor(np.load(f"../NormalizedQualityFiltered/{file_name}.npy")).unsqueeze(1)
inputs = vol[0:9].unsqueeze(0)
targets = vol[9:].unsqueeze(0)

outputs = pl_model.forward(inputs.to(device))
outputs = outputs.detach().cpu()
outputs = torch.concat([inputs, outputs], dim=1).squeeze(0, 2).numpy()
targets = torch.concat([inputs, targets], dim=1).squeeze(0, 2).numpy()
print(outputs.shape)
print(targets.shape)

np.save(f"outputs_{file_name}_{config["run_name"]}.npy", outputs)
np.save(f"targets_{file_name}_{config["run_name"]}.npy", targets)

sorted(dm.test_paths)

def multi_vol_seq_interactive(volume_seqs, titles=None):
    """
    Interactive plot of multiple volume sequences using ipywidgets
    
    Parameters:
    - volume_seqs: List of 4D volume sequences to display
    - titles: Optional list of titles for each sequence
    """
    print(len(volume_seqs))
    if titles is None:
        titles = [f"Volume {i+1}" for i in range(len(volume_seqs))]
        
    num_volumes = len(volume_seqs)
    nrows = int(num_volumes ** 0.5)
    ncols = (num_volumes + nrows - 1) // nrows
    
    def plot_volumes(time_idx, slice_idx):
        fig, axes = plt.subplots(nrows, ncols, 
                                figsize=(5*ncols, 5*nrows),
                                squeeze=True)
        if nrows == 1:
            if ncols == 1:
                axes = [[axes]]
            else:
                axes = [axes]
                
        for i, (volume_seq, title) in enumerate(zip(volume_seqs, titles)):
            row, col = i // ncols, i % ncols
            ax = axes[row][col]
            
            t = min(time_idx, len(volume_seq) - 1)
            s = min(slice_idx, len(volume_seq[t]) - 1)
            
            im = ax.imshow(volume_seq[t][s], cmap='magma')
            ax.set_title(title)
            plt.colorbar(im, ax=ax)
            
        plt.tight_layout()
        plt.show(block=True)
        
    max_time = max(len(vol) for vol in volume_seqs) - 1
    max_slice = max(len(vol[0]) for vol in volume_seqs) - 1
    
    interact(
        plot_volumes,
        time_idx=IntSlider(min=0, max=max_time, step=1, value=0, description='Time:'),
        slice_idx=IntSlider(min=0, max=max_slice, step=1, value=0, description='Slice:')
    )

multi_vol_seq_interactive([outputs, targets])